# assert checks

sourced from -- https://www.youtube.com/watch?v=5nXmq1PsoJ0

In [4]:
def gcd(a, b):
    assert isinstance(a, int), "argument a must be integer."
    assert isinstance(b, int), "arguments b must be integer."
    while b:
        a, b = b, a % b
    return a
# print(gcd(48, 18.1))  # fails with assertion
# print(gcd(48.1, 18.1))  # fails with assertion at runtime
print(gcd(48, 18))  # Output: 6

6


using type system to understand better

In [19]:
class Contract:
    def __set__(self, instance, value):
        self.check(value)
        instance.__dict__[self.name] = value
    def __set_name__(self, cls, name):
        self.name = name
    @classmethod
    def check(cls, value):
        pass

# standalone 
class isInteger(Contract):
    @classmethod
    def check(cls, value):
        assert isinstance(value, int), 'Expected int'

# improvement on above
class Typed(Contract):
    type = None
    @classmethod
    def check(cls, value):
        assert isinstance(value, cls.type), f'Expected {cls.type}'

class Integer(Typed):
    type = int

# Integer.check(2.2) # fails with assertion
class Float(Typed):
    type = float
# Float.check(2) # fails with assertion
class Positive(Contract):
    @classmethod
    def check(cls, value):
        assert value > 0, 'Must be greater than 0'
        super().check(value)
class PositiveInt(Positive, Integer):
    pass

def gcd_typed(a:Integer, b:Integer):
    Integer.check(a)
    Positive.check(a)
    PositiveInt.check(b)
    while(b):
        a, b = b, a % b
    return a

# print(gcd_typed(2.2, 3)) # fails with assertion
print(gcd_typed(2, 3)) 

from functools import wraps
from inspect import signature
sig = signature(gcd_typed)
print(sig)

def checked(func):
    sig = signature(func)
    ann = func.__annotations__
    @wraps(func)
    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        for name, val in bound.arguments.items():
            if name in ann:
                ann[name].check(val)
        return func(*args, **kwargs)
    return wrapper


@checked
def gcd_deco(a:PositiveInt, b:PositiveInt):
    while b:
        a, b = b, a % b
    return a

# gcd_deco(2.3, 2.8)   # gives "Expected <class 'int'>"
gcd_deco(16, 8)

1
(a: __main__.Integer, b: __main__.Integer)


8

# descriptor protocol

In [22]:
class NonEmptyString(Contract):
    @classmethod
    def check(cls, value):
        assert len(value) > 0, "Empty string .. expected string with non zero length"

class TypeBase:
    @classmethod
    def __init_subclass__(cls):
        for name, val in cls.__annotations__.items():
            contract = val()
            contract.__set_name__(cls, name)
            setattr(cls, name, contract)

class Player(TypeBase):
    name: NonEmptyString
    x: Integer
    y: Integer
    def __init__(self, name, x, y) -> None:
        self.name = name
        self.x, self.y = x, y
        super().__init__()
    def left(self, dx):
        self.x -= dx
    def right(self, dx):
        self.x += dx

p = Player('MyName', 0, 0)
# p.x = '23' -- Expected <class 'int'>
p.x = 23
